In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder,StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
import itertools
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,confusion_matrix,r2_score
import warnings
warnings.filterwarnings("ignore", message="X has feature names")


In [ ]:
columns = ["duration","protocol_type","service","flag","src_bytes",
    "dst_bytes","land","wrong_fragment","urgent","hot","num_failed_logins",
    "logged_in","num_compromised","root_shell","su_attempted","num_root",
    "num_file_creations","num_shells","num_access_files","num_outbound_cmds",
    "is_host_login","is_guest_login","count","srv_count","serror_rate",
    "srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate",
    "diff_srv_rate","srv_diff_host_rate","dst_host_count","dst_host_srv_count",
    "dst_host_same_srv_rate","dst_host_diff_srv_rate","dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate","dst_host_serror_rate","dst_host_srv_serror_rate",
    "dst_host_rerror_rate","dst_host_srv_rerror_rate","last_flag", "attack"]


In [ ]:
network_data=pd.read_csv('c:/Users/adith/OneDrive/Desktop/data/sub.xlsx')

In [ ]:
network_data.head()

In [ ]:
network_data.tail()

In [ ]:
network_data.info()

In [ ]:
network_data.describe()

In [ ]:
network_data.shape

In [ ]:
network_data.isnull().any()

In [ ]:
print(f"Number of duplicate rows: {network_data.duplicated().sum()}")

In [ ]:
network_data["class"].value_counts().plot(kind='bar', color=["salmon","lightblue"])  

In [ ]:

def le(df):
    for col in df.columns:
        if df[col].dtype == 'object':
                label_encoder = LabelEncoder()
                df[col] = label_encoder.fit_transform(df[col])

le(network_data)

In [ ]:
network_data.head()

In [ ]:
X = network_data.drop(columns=['class'])
y = network_data['class']

In [ ]:
rfc = RandomForestClassifier()

rfe = RFE(rfc, n_features_to_select=10)
rfe = rfe.fit( X , y)

feature_map = [(i, v) for i, v in itertools.zip_longest(rfe.get_support(), X.columns)]
selected_features = [v for i, v in feature_map if i==True]

selected_features

In [ ]:
['protocol_type',
 'service',
 'flag',
 'src_bytes',
 'dst_bytes',
 'count',
 'same_srv_rate',
 'dst_host_srv_count',
 'dst_host_same_srv_rate',
 'dst_host_same_src_port_rate']

In [ ]:
X= X[selected_features]

In [ ]:
X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_train.shape


In [ ]:
X_test.shape


In [ ]:
y_train.shape


In [ ]:

y_test.shape

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
model1=KNeighborsClassifier()
model2 = LogisticRegression()
model3 = DecisionTreeClassifier()

In [ ]:
print('KNN Model:')

#Train the model
model1.fit(X_train_scaled, y_train)

# Evaluate using k-fold cross-validation (k=10)
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
cv_scores = cross_val_score(model1, X_train_scaled, y_train, cv=kfold)
print(f"Accuracy (k-fold cross-validation): {np.mean(cv_scores):.4f}")


#Test the model
y_pred_knn = model1.predict(X_test_scaled)
accuracy_knn = accuracy_score(y_test, y_pred_knn)
print(f'Accuracy(KNN): {accuracy_knn}')

In [ ]:
cm = confusion_matrix(y_test, y_pred_knn)
print("Confusion Matrix:")
print(cm)


plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Model 1 Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
TN, FP, FN, TP = cm.ravel()
 # Precision: TP / (TP + FP)
precision = TP / (TP + FP)

    # Recall: TP / (TP + FN)
recall = TP / (TP + FN)

    # F1 Score: 2 * (Precision * Recall) / (Precision + Recall)
f1 = 2 * (precision * recall) / (precision + recall)




print("Precision:",precision)
print("Rcall:",recall)
print("F_score:",f1)






In [ ]:
print('Logistic Regression:')
# Train the model
model2.fit(X_train_scaled, y_train)
# Evaluate using k-fold cross-validation (k=10)
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
cv_scores = cross_val_score(model2, X_train_scaled, y_train, cv=kfold)
print(f"Accuracy (k-fold cross-validation): {np.mean(cv_scores):.4f}")


#Test the model
y_pred = model2.predict(X_test_scaled)
# Calculate evaluation metrics
accuracy_lg = accuracy_score(y_test, y_pred)
print(f"Accuracy (test set): {accuracy_lg:.4f}")

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Model 2 Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


In [ ]:
TN, FP, FN, TP = cm.ravel()
 # Precision: TP / (TP + FP)
precision = TP / (TP + FP)

    # Recall: TP / (TP + FN)
recall = TP / (TP + FN)

    # F1 Score: 2 * (Precision * Recall) / (Precision + Recall)
f1 = 2 * (precision * recall) / (precision + recall)




print("Precision:",precision)
print("Recall:",recall)
print("F_score:",f1)

# Calculate R-squared error
r_squared = r2_score(y_test, y_pred)
print(f"R-squared (test set): {r_squared:.4f}")


In [ ]:
print("\nDesicion Tree")
# Train the model
model3.fit(X_train_scaled, y_train)

# Evaluate using k-fold cross-validation (k=5)
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
cv_scores = cross_val_score(model1, X_train_scaled, y_train, cv=kfold)
print(f"Accuracy (k-fold cross-validation): {np.mean(cv_scores):.4f}")


# Test the model
y_pred = model3.predict(X_test_scaled)

accuracy_dt= accuracy_score(y_test, y_pred)
print(f"Accuracy (test set): {accuracy_dt:.4f}")



In [ ]:
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Model 3 Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


In [ ]:
TN, FP, FN, TP = cm.ravel()
 # Precision: TP / (TP + FP)
precision = TP / (TP + FP)

    # Recall: TP / (TP + FN)
recall = TP / (TP + FN)

    # F1 Score: 2 * (Precision * Recall) / (Precision + Recall)
f1 = 2 * (precision * recall) / (precision + recall)




print("Precision:",precision)
print("Rcall:",recall)
print("F_score:",f1)

In [ ]:
data = {'Model': ['KNN', 'Logistic Regression', 'Decision Tree'],
        'Accuracy': [accuracy_knn, accuracy_lg, accuracy_dt]}
accuracy_df = pd.DataFrame(data)

# Display the DataFrame as a table
print(accuracy_df)

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(accuracy_df['Model'], accuracy_df['Accuracy'], color='skyblue')
plt.xlabel('Model', fontsize=14)
plt.ylabel('Accuracy', fontsize=14)
plt.title('Accuracy Comparison of Different Models', fontsize=16)
plt.ylim(0, 1)  # Set y-axis limit from 0 to 1
plt.xticks(rotation=45, ha='right')
plt.tight_layout()  # Adjust layout to prevent clipping of labels
plt.show()

In [ ]:
# Sample unseen data with selected columns
unseen_data = {
    'protocol_type': [0, 2],
    'service': [14, 22],
    'flag': [8, 9],
    'src_bytes': [1032, 199],
    'dst_bytes': [0, 420],
    'count': [13, 2],
    'same_srv_rate': [1.00, 0.08], 
    'diff_srv_rate': [0, 0.05],  
    'dst_host_srv_count': [255, 13],  
    'dst_host_same_srv_rate': [1.00, 0]  
}

# Create DataFrame from unseen data
unseen_df = pd.DataFrame(unseen_data)

# Drop the last two rows
# Display the DataFrame with selected columns
print(unseen_df)

In [ ]:
print('Sample Unseen Data Predictions:')
unseen_pred_knn = model1.predict(unseen_df)
unseen_pred_lr = model2.predict(unseen_df)
unseen_pred_dt = model3.predict(unseen_df)

label_mapping = {1: 'normal', 0: 'attack'}

# Map the numerical labels to 'normal' or 'attack' for each model's predictions
unseen_pred_knn_labels = [label_mapping.get(label, 'unknown') for label in unseen_pred_knn]
unseen_pred_dt_labels = [label_mapping.get(label, 'unknown') for label in unseen_pred_dt]
unseen_pred_lr_labels = [label_mapping.get(label, 'unknown') for label in unseen_pred_lr]

# Display the predictions
prediction={
    'KNN Predictions': unseen_pred_knn_labels,
    'Decision Tree Predictions': unseen_pred_dt_labels,
    'Logistic Regression Predictions': unseen_pred_lr_labels
}
prediction_df=pd.DataFrame(prediction)

print(prediction_df)